# Unsupervised Learning — Clustering & Dimensionality Reduction

When you don't have labels, unsupervised learning finds structure in data. Customer segmentation, anomaly detection, topic modeling — all unsupervised.

**Topics:** K-Means, DBSCAN, Hierarchical Clustering, GMM, PCA, t-SNE, UMAP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, load_digits
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. K-Means Clustering

In [ ]:
# Generate dataset
X_blobs, y_true = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)
X_blobs_scaled = StandardScaler().fit_transform(X_blobs)

# Find optimal k with Elbow method + Silhouette score
k_range = range(2, 10)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs_scaled, labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Elbow plot
axes[0].plot(k_range, inertias, 'bo-', lw=2)
axes[0].axvline(4, color='red', ls='--', label='True k=4')
axes[0].set_xlabel('Number of clusters k'); axes[0].set_ylabel('Inertia (within-cluster SSE)')
axes[0].set_title('Elbow Method'); axes[0].legend(); axes[0].grid(True)

# Silhouette plot
axes[1].plot(k_range, silhouettes, 'rs-', lw=2)
best_k = k_range[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='red', ls='--', label=f'Best k={best_k}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher=better)'); axes[1].legend(); axes[1].grid(True)

# Final clustering
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_km = km_final.fit_predict(X_blobs_scaled)
axes[2].scatter(X_blobs[:,0], X_blobs[:,1], c=labels_km, cmap='tab10', s=20, alpha=0.8)
axes[2].scatter(km_final.cluster_centers_[:,0]*X_blobs_scaled.std()+X_blobs.mean(),
                km_final.cluster_centers_[:,1]*X_blobs_scaled.std()+X_blobs.mean(),
                marker='X', s=200, c='black', zorder=5, label='Centroids')
axes[2].set_title(f'K-Means (k=4)\nARI={adjusted_rand_score(y_true, labels_km):.3f}')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('K-Means Clustering Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. DBSCAN — Density-Based Clustering

In [ ]:
# DBSCAN: finds clusters of arbitrary shape, handles noise/outliers
# No need to specify k! But needs eps and min_samples tuning.
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
X_moons_scaled = StandardScaler().fit_transform(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# K-Means fails on non-convex shapes
km_moons = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moons_scaled)
axes[0].scatter(X_moons[:,0], X_moons[:,1], c=km_moons, cmap='RdBu', s=20, alpha=0.8)
axes[0].set_title(f'K-Means (fails on moons!)\nARI={adjusted_rand_score(y_moons, km_moons):.3f}')

# DBSCAN succeeds
db = DBSCAN(eps=0.3, min_samples=5)
labels_db = db.fit_predict(X_moons_scaled)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = (labels_db == -1).sum()
axes[1].scatter(X_moons[:,0], X_moons[:,1], c=labels_db, cmap='RdBu', s=20, alpha=0.8)
axes[1].set_title(f'DBSCAN (eps=0.3)\n{n_clusters} clusters, {n_noise} noise pts')

# Add outliers — DBSCAN handles them naturally
X_with_outliers = np.vstack([X_moons, np.random.uniform(-2, 3, (15, 2))])
labels_with_outliers = DBSCAN(eps=0.3, min_samples=5).fit_predict(
    StandardScaler().fit_transform(X_with_outliers))
scatter = axes[2].scatter(X_with_outliers[:,0], X_with_outliers[:,1], 
                          c=labels_with_outliers, cmap='tab10', s=20, alpha=0.8)
# Noise points (label=-1) in different color
noise_mask = labels_with_outliers == -1
axes[2].scatter(X_with_outliers[noise_mask,0], X_with_outliers[noise_mask,1], 
               c='black', s=40, marker='x', label='Noise/Outliers')
axes[2].set_title('DBSCAN with Outliers\n(black X = noise)'); axes[2].legend()

plt.suptitle('DBSCAN vs K-Means on Non-Convex Data', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Dimensionality Reduction — t-SNE & PCA

In [ ]:
# Use digits dataset: 64 features → 2D visualization
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
X_digits_scaled = StandardScaler().fit_transform(X_digits)

# PCA: linear, fast, preserves global structure
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_digits_scaled)

# t-SNE: nonlinear, slow, preserves local structure (better for visualization)
# Pro tip: run PCA first to speed up t-SNE
X_pca_50 = PCA(n_components=50, random_state=42).fit_transform(X_digits_scaled)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca_50)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for ax, (X_2d, title) in zip(axes, [
    (X_pca, f'PCA ({pca.explained_variance_ratio_.sum():.1%} variance)'),
    (X_tsne, 't-SNE (perplexity=30)'),
]):
    for digit in range(10):
        mask = y_digits == digit
        ax.scatter(X_2d[mask,0], X_2d[mask,1], c=[colors[digit]], 
                   s=15, alpha=0.7, label=str(digit))
    ax.set_title(title, fontsize=12)
    ax.legend(title='Digit', fontsize=7, markerscale=2, 
              loc='upper right', ncol=2, bbox_to_anchor=(1.15, 1))
    ax.grid(True, alpha=0.3)

plt.suptitle('Digits Dataset: PCA vs t-SNE (64D → 2D)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('PCA: linear, interpretable, fast, good for preprocessing')
print('t-SNE: nonlinear, beautiful clusters, slow, good ONLY for visualization (not for ML features!)')
print('UMAP: modern alternative — faster than t-SNE, preserves global structure better')